In [23]:
import random
from math import log2
from module.conf import PROJECT_DIR

In [24]:
# Read CSV
def load_csv(filename):
    data = []
    with open(filename, 'r', encoding='utf-8') as file:
        lines = file.readlines()
        for line in lines[1:]:  # Ignore header
            category, message = line.strip().split(',', 1)
            message = message.strip('"')  # remove ()
            data.append([category, message])
    return data

### 1. load data

In [25]:
data = load_csv(filename=f"{PROJECT_DIR}/data/basic/email/spam.csv")
data[:5]

[['ham',
  'Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...'],
 ['ham', 'Ok lar... Joking wif u oni...'],
 ['spam',
  "Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's"],
 ['ham', 'U dun say so early hor... U c already then say...'],
 ['ham', "Nah I don't think he goes to usf, he lives around here though"]]

In [26]:
# Extract keywords in msg
def extract_keywords(data, top_n=20):
    word_freq = {}
    for _, message in data:
        words = message.lower() \
            .replace(',', ' ') \
            .replace('.', ' ') \
            .replace('!', ' ') \
            .replace('(', '').replace(')', '') \
            .replace('[', '').replace(']', '') \
            .split()
        for word in words:
            word_freq[word] = word_freq.get(word, 0) + 1

    # sort by frequency and get top N
    sorted_words = sorted(word_freq.items(), key=lambda x: x[1], reverse=True)
    keywords = [word for word, _ in sorted_words[:top_n]]
    return keywords


# Extract features from msg
def extract_features(message, keywords):
    message = message.lower()
    features = []
    for keyword in keywords:
        count = message.count(keyword)
        features.append(count)
    return features


# Change data to number
def preprocess_data(data, keywords):
    processed_data = []
    for category, message in data:
        features = extract_features(message, keywords)
        label = 1 if category == 'spam' else 0  # spam: 1, ham: 0
        processed_data.append(features + [label])
    return processed_data

In [27]:
test_keywords = extract_keywords(data, 200)
# prepared_data = preprocess_data(data, test_keywords)

In [28]:
class Node():
    def __init__(self, feature: int = None, threshold: float = None, left: list = None, right: list = None,
                 value: float = None):
        """
        Decision Tree node
        :param feature: feature index
        :param threshold: threshold to split left and right
        :param left: left list
        :param right: right list
        :param value: majority vote for leaf nodes
        """
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value
        return

$$
\begin{align}
p_i &= \frac{N_i}{N} \\
Entropy(node) &= -\sum_{i=1}^{n} p_i\log_2(p_i) \\
InformationGain &= Entropy(parent) - \sum_{child} \frac{N_{child}}{N} Entropy(child) \\
\end{align}
$$

In [29]:
def calc_entropy(labels) -> float:
    """
    calculate entropy of labels
    :param labels: labels need to calculate entropy
    :return: entropy
    """
    n = len(labels)
    if n == 0:
        return 0
    unq_labels = set(labels)
    entropy = 0
    for label in unq_labels:
        p = labels.count(label) / n
        entropy -= p * log2(p)
        pass
    return entropy


def calc_information_gain(data, feature_idx, threshold) -> (float, list, list):
    """
    calculate information gain
    :param data: (n, m) matrix
    :param feature_idx: index of feature
    :param threshold: threshold of entropy
    :return: inf_gain, left, right matched
    """
    n = len(data)
    if n == 0: return (0, [], [])

    labels = [row[-1] for row in data]
    parent_entropy = calc_entropy(labels)

    left = [row for row in data if row[feature_idx] <= threshold]
    right = [row for row in data if row[feature_idx] > threshold]

    left_entropy = calc_entropy([row[-1] for row in left])
    right_entropy = calc_entropy([row[-1] for row in right])

    branch_entropies = len(left) / len(data) * left_entropy + len(right) / len(data) * right_entropy
    inf_gain = parent_entropy - branch_entropies

    return inf_gain, left, right

In [30]:
def find_best_split(data) -> (int, float, float, list, list):
    """
    find best split, maximum information gain
    :param data: (n, m) with data[:-1] is labels
    :return: best (feature index, threshold, gain, left, right)
    """
    max_gain = -1
    best_feature_idx = None
    best_threshold = None
    best_left = None
    best_right = None

    n_features = len(data[0]) - 1
    for feature_idx in range(n_features):
        # find and sort value by feature_idx
        values = sorted(set(row[feature_idx] for row in data))
        for val, next_val in zip(values[:-1], values[1:]):
            threshold = (val + next_val) / 2
            inf_gain, left, right = calc_information_gain(data, feature_idx, threshold)
            if inf_gain > max_gain:
                max_gain = inf_gain
                best_feature_idx = feature_idx
                best_threshold = threshold
                best_left = left
                best_right = right
                pass
            pass
        pass
    return best_feature_idx, best_threshold, max_gain, best_left, best_right


def build_tree(data, curr_depth=0, max_depth=1) -> Node:
    """
    build decision tree
    :param data: data with data[:-1] is labels
    :param curr_depth: current depth
    :param max_depth: max depth
    :return: tree node
    """
    labels = [row[-1] for row in data]
    # purity = 1
    if len(set(labels)) == 1:  # same label
        return Node(value=labels[0])
    # reach max_depth
    if curr_depth >= max_depth:
        # find most frequent elements
        majority = max(set(labels), key=labels.count)
        return Node(value=majority)
    # find best split
    feature_idx, threshold, gain, left, right = find_best_split(data)

    # no gain
    if gain <= 0:
        majority = max(set(labels), key=labels.count)
        return Node(value=majority)
    # gain
    left_node = build_tree(left, curr_depth + 1, max_depth)
    right_node = build_tree(right, curr_depth + 1, max_depth)
    return Node(feature=feature_idx, threshold=threshold, left=left_node, right=right_node)

In [31]:
def predict(tree, sample) -> float:
    """
    predict
    :param tree: decision tree root node
    :param sample: sample data
    :return: value to predict
    """
    if tree.value is not None:
        return tree.value
    if sample[tree.feature] <= tree.threshold:
        return predict(tree.left, sample)
    else:
        return predict(tree.right, sample)

In [32]:
def train_test_split(data, test_size=0.2):
    n = len(data)
    n_test = int(n * test_size)
    test_indices = set(random.sample(range(n), n_test))
    train_data = [data[i] for i in range(n) if i not in test_indices]
    test_data = [data[i] for i in range(n) if i in test_indices]
    return train_data, test_data

In [33]:
def print_tree(keywords, node, indent=""):
    if node.value is not None:
        print(f"{indent}Leaf: {int(node.value)}")
    else:
        print(f"{indent}Feature '{keywords[node.feature]}' count <= {node.threshold}")
        print_tree(keywords, node.left, indent + "L:")
        print_tree(keywords, node.right, indent + "R:")
    return


def main():
    filename = f"{PROJECT_DIR}/data/basic/email/spam.csv"
    raw_data = load_csv(filename)
    print(f"Loaded {len(raw_data)} samples from {filename}")

    keywords = extract_keywords(raw_data, top_n=2000)  # get top 200 keywords
    print(f"Feature keywords: {keywords}")

    data = preprocess_data(raw_data, keywords)

    train_data, test_data = train_test_split(data, test_size=0.2)
    print(f"Train set: {len(train_data)} samples")
    print(f"Test set: {len(test_data)} samples")

    max_depth = 5
    tree = build_tree(train_data, max_depth=max_depth)

    print("\nDecision Tree structure:")
    print_tree(keywords, tree)

    correct = 0
    for sample in test_data:
        pred = predict(tree, sample[:-1])
        if pred == sample[-1]:
            correct += 1
    accuracy = correct / len(test_data)
    print(f"\nTest Accuracy: {accuracy * 100:.2f}%")

    # predict a new message:
    new_message = "Free tickets to win a prize! Call now!"
    new_sample = extract_features(new_message, keywords)
    prediction = predict(tree, new_sample)
    print(f"Predict new message: '{new_message}': {'spam' if prediction == 1 else 'ham'}")
    return


# if __name__ == "__main__":
main()

Loaded 5574 samples from /Users/hiepnq/Working/training/python/learn-python/data/basic/email/spam.csv
Feature keywords: ['i', 'to', 'you', 'a', 'the', 'u', 'and', 'is', 'in', 'me', 'my', 'for', 'your', 'it', 'of', 'call', 'have', 'on', 'that', '2', 'are', 'now', 'so', 'but', 'not', 'or', 'can', 'at', 'do', 'will', "i'm", 'ur', 'be', 'if', 'get', 'with', 'just', 'we', 'this', 'no', 'up', 'when', 'from', '4', 'go', '&lt;#&gt;', 'ok', 'free', 'all', 'how', 'out', 'what', 'know', 'like', 'then', 'good', 'got', 'was', 'come', 'am', 'its', 'love', 'time', 'only', '?', 'day', 'send', 'he', 'there', 'want', 'text', 'as', 'by', 'one', "i'll", 'need', 'ü', 'home', 'going', 'about', 'lor', 'sorry', 'see', 'still', 'txt', 'r', 'n', 'reply', 'dont', 'back', 'our', 'she', 'stop', "don't", 'tell', 'mobile', 'new', 'take', 'hi', 'da', 'any', 'today', 'please', 'pls', 'think', 'been', 'they', 'her', 'later', 'k', 'did', 'dear', 'phone', 'some', 'has', 'well', 'great', 'an', 'hey', 'here', 'claim', 'hop